# Train the captioning student — LoRA fine-tune Qwen2.5-VL-3B (Colab GPU)

**Distillation**: the teacher (Qwen3-VL-30B) already captioned the clips; here we
LoRA-fine-tune a small **Qwen2.5-VL-3B-Instruct** *student* to reproduce those
captions from a few frames per clip, so you get a cheap captioner you can run yourself.

- Data: the clips + `octopus_clips_verified.json` (each entry's `caption`) you uploaded to Drive.
- Input per clip: `N_FRAMES` sampled frames. Target: the entry's caption
  (clips labeled `octopus not present` are kept but subsampled so they don't dominate).
- Method: 4-bit QLoRA (fits A100-40GB comfortably; also L4-24GB).

> Runtime → A100 GPU. This is a real fine-tune — expect ~30–90 min depending on data size.

## 1. Install

In [ ]:
!pip -q install -U "transformers>=4.49" "trl>=0.12" peft accelerate bitsandbytes qwen-vl-utils
!apt-get -qq install -y ffmpeg >/dev/null
print("Installed. If a CUDA/torch error appears later: Runtime -> Restart session, then re-run from CONFIG.")

In [ ]:
import torch
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")

## 2. Config

In [ ]:
from pathlib import Path

BASE_MODEL   = "Qwen/Qwen2.5-VL-3B-Instruct"   # student (teacher was Qwen3-VL-30B)
N_FRAMES     = 4          # frames sampled per clip (kept small: multi-image => long sequences)
MAX_PIXELS   = 360*420    # per-frame cap (memory); raise if VRAM allows
MAX_ABSENT   = 120        # cap 'octopus not present' training clips so they don't dominate
VAL_FRAC     = 0.12

# LoRA / QLoRA
USE_4BIT     = True
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
EPOCHS       = 3
LR           = 1e-4
BATCH        = 1
GRAD_ACCUM   = 8

DRIVE_ROOT   = Path("/content/drive/MyDrive/GSOC-Catrobat")
INDEX_JSON   = Path("octopus_clips_verified.json")   # the captions you uploaded (set to -2 if that's the file)
CLIPS_ROOT   = Path("octopus_clips_verified")
OUT_DIR      = Path("caption_student_qwen25vl3b_lora")
FRAMES_CACHE = Path("frames_cache"); FRAMES_CACHE.mkdir(exist_ok=True)
PROMPT = ("These frames are sampled in order from one short aquarium clip of Nity, an octopus. "
          "Write ONE sentence describing what the octopus does across the clip, "
          "or 'octopus not present' if no octopus is visible.")

## 3. Get data from Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import zipfile, shutil
with zipfile.ZipFile(DRIVE_ROOT / "octopus_clips_verified.zip") as z:
    z.extractall(".")
shutil.copy(DRIVE_ROOT / INDEX_JSON.name, INDEX_JSON)
print(len(list(CLIPS_ROOT.rglob("*.mp4"))), "clips |", INDEX_JSON)

## 4. Build (frames -> caption) dataset

In [ ]:
import json, subprocess
from pathlib import Path as P
import numpy as np

def resolve(cp): return CLIPS_ROOT / cp.split("octopus_clips_verified/",1)[-1]

def sample_frames(clip_path, stem):
    outdir = FRAMES_CACHE / stem; outdir.mkdir(exist_ok=True)
    got = sorted(outdir.glob("f_*.jpg"))
    if not got:
        subprocess.run(["ffmpeg","-y","-loglevel","error","-i",str(clip_path),
                        "-vf","fps=1,scale='min(640,iw)':-2","-q:v","3",str(outdir/"f_%03d.jpg")],
                       capture_output=True)
        got = sorted(outdir.glob("f_*.jpg"))
    if not got: return []
    idx = np.linspace(0, len(got)-1, min(N_FRAMES, len(got))).round().astype(int)
    return [str(got[i]) for i in idx]

index = json.load(open(INDEX_JSON))["clips"]
samples, absent = [], 0
for e in index:
    cap = (e.get("caption") or "").strip()
    if not cap: continue
    cp = resolve(e["clip_path"])
    if not cp.exists(): continue
    if cap.lower() == "octopus not present":
        if absent >= MAX_ABSENT: continue
        absent += 1
    frames = sample_frames(cp, cp.stem + "_" + e.get("segment",""))
    if not frames: continue
    content = [{"type":"image","image":f,"max_pixels":MAX_PIXELS} for f in frames]
    content.append({"type":"text","text":PROMPT})
    samples.append({"messages":[
        {"role":"user","content":content},
        {"role":"assistant","content":[{"type":"text","text":cap}]}]})

import random; random.seed(42); random.shuffle(samples)
nval = max(1, int(len(samples)*VAL_FRAC))
val, train = samples[:nval], samples[nval:]
print(f"train {len(train)} | val {len(val)} | absent kept {absent}")

## 5. Load Qwen2.5-VL-3B + LoRA

In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

processor = AutoProcessor.from_pretrained(BASE_MODEL, max_pixels=MAX_PIXELS)
qcfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                          bnb_4bit_compute_dtype=torch.bfloat16) if USE_4BIT else None
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    BASE_MODEL, torch_dtype=torch.bfloat16, quantization_config=qcfg, device_map="auto")
if USE_4BIT: model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

lora = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none",
                  task_type="CAUSAL_LM",
                  target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 6. Collator (mask prompt + image tokens; train only on the caption)

In [ ]:
from qwen_vl_utils import process_vision_info

IMG_TOKEN_ID = model.config.image_token_id if hasattr(model.config,"image_token_id") else \
               processor.tokenizer.convert_tokens_to_ids("<|image_pad|>")

def collate(examples):
    texts, imgs = [], []
    for ex in examples:
        texts.append(processor.apply_chat_template(ex["messages"], tokenize=False, add_generation_prompt=False))
        im,_ = process_vision_info(ex["messages"]); imgs.append(im)
    batch = processor(text=texts, images=imgs, return_tensors="pt", padding=True)
    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    labels[labels == IMG_TOKEN_ID] = -100
    batch["labels"] = labels
    return batch

## 7. Train

In [ ]:
from trl import SFTTrainer, SFTConfig

args = SFTConfig(
    output_dir=str(OUT_DIR), num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR, lr_scheduler_type="cosine", warmup_ratio=0.05,
    logging_steps=10, save_strategy="epoch", eval_strategy="epoch",
    bf16=True, gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    remove_unused_columns=False, dataset_kwargs={"skip_prepare_dataset": True},
    report_to="none")

trainer = SFTTrainer(model=model, args=args, train_dataset=train, eval_dataset=val,
                     data_collator=collate)
trainer.train()
trainer.save_model(str(OUT_DIR))
processor.save_pretrained(str(OUT_DIR))
print("saved LoRA adapter ->", OUT_DIR)

## 8. Quick check + save adapter to Drive

In [ ]:
import torch
model.eval()
for ex in val[:5]:
    msgs=[ex["messages"][0]]
    text=processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    im,_=process_vision_info(msgs)
    inp=processor(text=[text], images=[im], return_tensors="pt").to(model.device)
    inp.pop("mm_token_type_ids", None)   # generate() rejects this key in some transformers versions
    with torch.no_grad():
        out=model.generate(**inp, max_new_tokens=80, do_sample=False)
    pred=processor.batch_decode(out[:, inp["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()
    ref=ex["messages"][1]["content"][0]["text"]
    print("PRED:",pred); print("REF :",ref); print("-"*50)

In [ ]:
import shutil
shutil.make_archive(str(DRIVE_ROOT/OUT_DIR.name), "zip", OUT_DIR)
print("Adapter zipped to Drive:", DRIVE_ROOT/(OUT_DIR.name+".zip"))

## 9. Release the Colab GPU (run LAST)
Frees VRAM and **unassigns the runtime** so the GPU is released. Run only after the adapter is saved to Drive (cell 8) — nothing after `runtime.unassign()` executes.

In [ ]:
# === Release the Colab GPU (run LAST — after the adapter zip is saved to Drive) ===
import gc, torch
for v in ["trainer", "model", "inp", "out"]:
    if v in globals():
        del globals()[v]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"VRAM still allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# Disconnects the runtime and frees the GPU. Nothing after this runs.
from google.colab import runtime
runtime.unassign()